In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS sf_fire_bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS sf_fire_silver")
spark.sql("CREATE DATABASE IF NOT EXISTS sf_fire_gold")

APP_TOKEN = "dUaoeepE9uZQep2gJD2fNSA2F"
BASE_URL = "https://data.sfgov.org/resource"

In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

def ingest_socrata_to_bronze(dataset_id, table_name, limit=50000):
    all_records = []
    offset = 0

    print(f"Ingesting {table_name}...")

    while True:
        url = f"{BASE_URL}/{dataset_id}.json"
        params = {
            "$$app_token": APP_TOKEN,
            "$limit": limit,
            "$offset": offset,
            "$order": ":id"
        }

        response = requests.get(url, params=params)
        records = response.json()

        if not records:
            break

        all_records.extend(records)
        offset += limit
        print(f"  {table_name}: {len(all_records)} records fetched...")

        # Cap for speed — enough for eval
        if len(all_records) >= 200000:
            break

    pdf = pd.DataFrame(all_records)
    df = spark.createDataFrame(pdf)
    df = df.withColumn("_ingested_at", current_timestamp()) \
           .withColumn("_source_id", lit(dataset_id))

    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"sf_fire_bronze.{table_name}")

    print(f"✅ {table_name}: {df.count()} records written to Bronze\n")

# Run all 5
ingest_socrata_to_bronze("wr8u-xric", "incidents")
ingest_socrata_to_bronze("nuek-vuh3", "calls_for_service")
ingest_socrata_to_bronze("da97-k3b4", "violations")
ingest_socrata_to_bronze("wb4c-6hwj", "inspections")
ingest_socrata_to_bronze("pwi8-y8qd", "permits")

In [0]:
# Clean up partial violations table if it exists
spark.sql("DROP TABLE IF EXISTS sf_fire_bronze.violations")
print("🗑️ Cleared partial violations table")

# Now ingest only the 3 remaining datasets with correct IDs
ingest_socrata_to_bronze("4zuq-2cbe", "violations")
ingest_socrata_to_bronze("wb4c-6hwj", "inspections")
ingest_socrata_to_bronze("893e-xam6", "permits")

end of bronze layer

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:

df = spark.table("sf_fire_bronze.incidents")
df.printSchema()


In [0]:
def clean_incidents():
    df = spark.table("sf_fire_bronze.incidents")

    df_clean = df \
        .withColumn("incident_date", to_timestamp("incident_date")) \
        .withColumn("alarm_dttm", to_timestamp("alarm_dttm")) \
        .withColumn("arrival_dttm", to_timestamp("arrival_dttm")) \
        .withColumn("close_dttm", to_timestamp("close_dttm")) \
        .withColumn("response_time_minutes",
            (unix_timestamp("arrival_dttm") - unix_timestamp("alarm_dttm")) / 60) \
        .withColumn("estimated_property_loss",
            when(col("estimated_property_loss").isNull(), 0)
            .otherwise(col("estimated_property_loss").cast(DoubleType()))) \
        .withColumn("estimated_contents_loss",
            when(col("estimated_contents_loss").isNull(), 0)
            .otherwise(col("estimated_contents_loss").cast(DoubleType()))) \
        .withColumn("suppression_units",
            col("suppression_units").cast(IntegerType())) \
        .withColumn("fire_fatalities",
            col("fire_fatalities").cast(IntegerType())) \
        .withColumn("civilian_fatalities",
            col("civilian_fatalities").cast(IntegerType())) \
        .withColumn("fire_injuries",
            col("fire_injuries").cast(IntegerType())) \
        .withColumn("civilian_injuries",
            col("civilian_injuries").cast(IntegerType())) \
        .filter(col("incident_date").isNotNull()) \
        .filter(col("arrival_dttm").isNotNull()) \
        .filter(col("response_time_minutes") > 0) \
        .filter(col("response_time_minutes") < 120) \
        .withColumn("address_clean", upper(trim(col("address")))) \
        .dropDuplicates(["incident_number"])

    df_clean.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("sf_fire_silver.incidents")

    print(f"✅ Incidents Silver: {df_clean.count()} records")

clean_incidents()

In [0]:
datasets = ["incidents", "calls_for_service", "violations", "inspections", "permits"]

for table in datasets:
    print(f"\n{'='*50}")
    print(f"SCHEMA: sf_fire_bronze.{table}")
    print('='*50)
    spark.table(f"sf_fire_bronze.{table}").printSchema()
    

In [0]:
def clean_calls():
    df = spark.table("sf_fire_bronze.calls_for_service")

    df_clean = df \
        .withColumn("received_dttm", to_timestamp("received_dttm")) \
        .withColumn("entry_dttm", to_timestamp("entry_dttm")) \
        .withColumn("dispatch_dttm", to_timestamp("dispatch_dttm")) \
        .withColumn("response_dttm", to_timestamp("response_dttm")) \
        .withColumn("on_scene_dttm", to_timestamp("on_scene_dttm")) \
        .withColumn("available_dttm", to_timestamp("available_dttm")) \
        .withColumn("dispatch_delay_seconds",
            unix_timestamp("dispatch_dttm") - unix_timestamp("received_dttm")) \
        .withColumn("response_delay_seconds",
            unix_timestamp("on_scene_dttm") - unix_timestamp("dispatch_dttm")) \
        .withColumn("call_duration_minutes",
            (unix_timestamp("available_dttm") - unix_timestamp("received_dttm")) / 60) \
        .withColumn("final_priority", col("final_priority").cast(IntegerType())) \
        .withColumn("number_of_alarms", col("number_of_alarms").cast(IntegerType())) \
        .withColumn("address_clean", upper(trim(col("address")))) \
        .filter(col("received_dttm").isNotNull()) \
        .filter(col("dispatch_delay_seconds") >= 0) \
        .filter(col("response_delay_seconds") >= 0) \
        .dropDuplicates(["call_number", "unit_id"])

    df_clean.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("sf_fire_silver.calls_for_service")

    print(f"✅ Calls Silver: {df_clean.count()} records")

clean_calls()

In [0]:
def clean_violations():
    df = spark.table("sf_fire_bronze.violations")

    df_clean = df \
        .withColumn("violation_date", to_timestamp("violation_date")) \
        .withColumn("close_date", to_timestamp("close_date")) \
        .withColumn("is_open",
            when(col("status") == "Open", 1).otherwise(0)) \
        .withColumn("days_since_violation",
            datediff(current_date(), col("violation_date"))) \
        .withColumn("days_to_close",
            when(col("close_date").isNotNull(),
                datediff(col("close_date"), col("violation_date")))
            .otherwise(None)) \
        .withColumn("address_clean", upper(trim(col("address")))) \
        .filter(col("violation_date").isNotNull()) \
        .dropDuplicates(["violation_number"])

    df_clean.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("sf_fire_silver.violations")

    print(f"✅ Violations Silver: {df_clean.count()} records")

clean_violations()

In [0]:
def clean_inspections():
    df = spark.table("sf_fire_bronze.inspections")

    df_clean = df \
        .withColumn("inspection_start_date", to_timestamp("inspection_start_date")) \
        .withColumn("inspection_end_date", to_timestamp("inspection_end_date")) \
        .withColumn("inspection_duration_days",
            datediff(col("inspection_end_date"), col("inspection_start_date"))) \
        .withColumn("invoice_amount",
            when(col("invoice_amount").isNull(), 0)
            .otherwise(col("invoice_amount").cast(DoubleType()))) \
        .withColumn("penalty_amount",
            when(col("penalty_amount").isNull(), 0)
            .otherwise(col("penalty_amount").cast(DoubleType()))) \
        .withColumn("passed_flag",
            when(col("inspection_status") == "completed", 1).otherwise(0)) \
        .withColumn("has_violation",
            when(col("violation_number").isNotNull(), 1).otherwise(0)) \
        .withColumn("address_clean", upper(trim(col("address")))) \
        .filter(col("inspection_start_date").isNotNull()) \
        .dropDuplicates(["inspection_number"])

    df_clean.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("sf_fire_silver.inspections")

    print(f"✅ Inspections Silver: {df_clean.count()} records")

clean_inspections()

In [0]:
def clean_permits():
    df = spark.table("sf_fire_bronze.permits")

    df_clean = df \
        .withColumn("permit_application_date", to_timestamp("permit_application_date")) \
        .withColumn("permit_date_approved", to_timestamp("permit_date_approved")) \
        .withColumn("expiration_date", to_timestamp("expiration_date")) \
        .withColumn("inspection_date", to_timestamp("inspection_date")) \
        .withColumn("is_active",
            when(col("permit_status") == "Active", 1).otherwise(0)) \
        .withColumn("is_expired",
            when(col("permit_status") == "Expired", 1).otherwise(0)) \
        .withColumn("days_until_expiry",
            datediff(col("expiration_date"), current_date())) \
        .withColumn("permit_fee",
            when(col("permit_fee").isNull(), 0)
            .otherwise(col("permit_fee").cast(DoubleType()))) \
        .withColumn("address_clean", upper(trim(col("permit_address")))) \
        .filter(col("permit_application_date").isNotNull()) \
        .dropDuplicates(["permit_number"])

    df_clean.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("sf_fire_silver.permits")

    print(f"✅ Permits Silver: {df_clean.count()} records")

clean_permits()

In [0]:
from pyspark.sql.functions import (
    count, sum, avg, min, max, when, col, datediff, 
    current_date, unix_timestamp, upper, trim
)
from pyspark.sql.types import DoubleType, IntegerType

In [0]:
# Run this first
for table in ["incidents", "violations", "inspections", "permits"]:
    print(f"\n{'='*50}")
    print(f"SCHEMA: sf_fire_silver.{table}")
    print('='*50)
    spark.table(f"sf_fire_silver.{table}").printSchema()
    

In [0]:
from pyspark.sql.functions import col, count, sum, avg, min, max, when, coalesce, lit, round

def build_gold_property_table():

    incidents  = spark.table("sf_fire_silver.incidents")
    violations = spark.table("sf_fire_silver.violations")
    inspections = spark.table("sf_fire_silver.inspections")
    permits    = spark.table("sf_fire_silver.permits")

    # -------------------------------
    # ⚠️ Violations
    # -------------------------------
    viol_agg = violations.groupBy("address_clean").agg(
        count("violation_number").alias("total_violations"),
        sum("is_open").alias("open_violations")
    ).withColumn("open_violation_ratio",
        col("open_violations") / (col("total_violations") + 1))

    # -------------------------------
    # 🚨 Incidents
    # -------------------------------
    inc_agg = incidents.groupBy("address_clean").agg(
        count("incident_number").alias("incident_history_count"),
        sum("estimated_property_loss").alias("total_property_loss"),
        avg("response_time_minutes").alias("avg_response_time"),
        sum(coalesce(col("fire_fatalities"), lit(0))).alias("total_fire_fatalities")
    )

    # -------------------------------
    # 🧪 Inspections
    # -------------------------------
    insp_agg = inspections.groupBy("address_clean").agg(
        count("inspection_number").alias("total_inspections"),
        avg("passed_flag").alias("pass_rate"),
        sum("has_violation").alias("inspections_with_violations")
    )

    # -------------------------------
    # 📜 Permits
    # -------------------------------
    perm_agg = permits.groupBy("address_clean").agg(
        count("permit_number").alias("total_permits"),
        sum("is_expired").alias("expired_permits")
    ).withColumn("expired_permit_ratio",
        col("expired_permits") / (col("total_permits") + 1))

    # -------------------------------
    # 🔗 Join
    # -------------------------------
    gold = viol_agg \
        .join(inc_agg, "address_clean", "left") \
        .join(insp_agg, "address_clean", "left") \
        .join(perm_agg, "address_clean", "left") \
        .fillna(0)

    # -------------------------------
    # 🧠 Risk Score
    # -------------------------------
    gold = gold.withColumn(
        "risk_score",
        round(
            col("open_violation_ratio") * 30 +
            col("incident_history_count") * 10 +
            col("expired_permit_ratio") * 20 +
            (1 - col("pass_rate")) * 20 +
            col("inspections_with_violations") * 5 +
            col("total_fire_fatalities") * 15,
            2
        )
    )

    # -------------------------------
    # 🎯 Risk Tier
    # -------------------------------
    gold = gold.withColumn(
        "risk_tier",
        when(col("risk_score") >= 60, "HIGH")
        .when(col("risk_score") >= 30, "MEDIUM")
        .otherwise("LOW")
    )

    # -------------------------------
    # 🧠 Binary Label (Optional)
    # -------------------------------
    gold = gold.withColumn(
        "high_risk_label",
        when(col("risk_tier") == "HIGH", 1).otherwise(0)
    )

    # -------------------------------
    # 💾 Save
    # -------------------------------
    gold.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sf_fire_gold.property_risk_master")

    # -------------------------------
    # 📊 Clean Output
    # -------------------------------
    print("\n" + "="*60)
    print("🔥 PROPERTY RISK SUMMARY")
    print("="*60)

    gold.groupBy("risk_tier").count().show()

    print("\n🏆 Top 10 High Risk Properties:")
    gold.orderBy(col("risk_score").desc()).select(
        "address_clean", "risk_score", "risk_tier"
    ).show(10, truncate=False)


# RUN
build_gold_property_table()

In [0]:
from pyspark.sql.functions import col, count, sum, avg, max, round

def build_gold_district_risk():

    df = spark.table("sf_fire_gold.property_risk_scores")

    district_summary = df.groupBy("neighborhood_district", "battalion").agg(
        count("*").alias("total_properties"),
        sum("high_risk_label").alias("high_risk_properties"),
        avg("risk_score").alias("avg_risk_score"),
        max("risk_score").alias("max_risk_score"),
        sum("total_violations").alias("district_total_violations"),
        sum("open_violations").alias("district_open_violations"),
        sum("incident_history_count").alias("district_total_incidents"),
        sum("total_fire_fatalities").alias("district_fire_fatalities"),
        sum("total_property_loss").alias("district_property_loss"),
        avg("avg_response_time").alias("district_avg_response_time")
    ).withColumn(
        "high_risk_pct",
        round(col("high_risk_properties") / col("total_properties") * 100, 2)
    )

    district_summary.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("sf_fire_gold.district_risk_summary")

        # -------------------------------
    # 📊 CLEAN DEMO OUTPUT
    # -------------------------------
    print("\n" + "="*70)
    print("🗺️ DISTRICT RISK INTELLIGENCE DASHBOARD")
    print("="*70)

    total_districts = district_summary.count()

    print("\n📊 OVERVIEW")
    print("-"*70)
    print(f"Total Districts: {total_districts}")

    # -------------------------------
    # 🔥 TOP HIGH RISK DISTRICTS
    # -------------------------------
    print("\n🔥 TOP 10 HIGH-RISK DISTRICTS")
    print("-"*70)

    district_summary.orderBy(col("avg_risk_score").desc()) \
        .select(
            "neighborhood_district",
            "battalion",
            "total_properties",
            "high_risk_properties",
            "high_risk_pct",
            "avg_risk_score"
        ).show(10, truncate=False)

    # -------------------------------
    # ⚠️ MOST INCIDENT-HEAVY DISTRICTS
    # -------------------------------
    print("\n🚨 TOP 10 INCIDENT-HEAVY DISTRICTS")
    print("-"*70)

    district_summary.orderBy(col("district_total_incidents").desc()) \
        .select(
            "neighborhood_district",
            "district_total_incidents",
            "district_fire_fatalities",
            "district_property_loss"
        ).show(10, truncate=False)

    # -------------------------------
    # 🚒 SLOW RESPONSE DISTRICTS
    # -------------------------------
    print("\n🚒 TOP 10 SLOW RESPONSE DISTRICTS")
    print("-"*70)

    district_summary.orderBy(col("district_avg_response_time").desc()) \
        .select(
            "neighborhood_district",
            "district_avg_response_time",
            "total_properties"
        ).show(10, truncate=False)

    # -------------------------------
    # 📊 RISK DISTRIBUTION
    # -------------------------------
    print("\n📊 RISK LEVEL DISTRIBUTION (BY %)")
    print("-"*70)

    district_summary.select("high_risk_pct") \
        .summary("min", "25%", "50%", "75%", "max") \
        .show()

    print("\n" + "="*70)
    print("✅ DISTRICT RISK ANALYSIS COMPLETE")
    print("="*70)

build_gold_district_risk()

In [0]:
from pyspark.sql.functions import (
    col, count, sum, avg, when, round,
    coalesce, lit, max, dense_rank
)
from pyspark.sql.window import Window

def build_gold_compliance():

    # -------------------------------
    # 📥 Load Silver Tables
    # -------------------------------
    violations  = spark.table("sf_fire_silver.violations")
    inspections = spark.table("sf_fire_silver.inspections")

    # -------------------------------
    # 🔗 District Mapping (for partitioning)
    # -------------------------------
    district_map = violations.select(
        "address_clean", "neighborhood_district"
    ).dropDuplicates(["address_clean"])

    # -------------------------------
    # 🔴 Violations Aggregation
    # -------------------------------
    viol = violations.groupBy("address_clean").agg(
        count("violation_number").alias("total_violations"),
        sum("is_open").alias("open_violations"),
        sum(when(col("status") == "Closed", 1).otherwise(0)).alias("closed_violations"),
        avg("days_to_close").alias("avg_days_to_resolve"),
        max("days_since_violation").alias("max_violation_age")
    )

    # -------------------------------
    # 🟢 Inspection Aggregation
    # -------------------------------
    insp = inspections.groupBy("address_clean").agg(
        count("inspection_number").alias("total_inspections"),
        avg("passed_flag").alias("inspection_pass_rate"),
        sum("has_violation").alias("failed_inspections"),
        sum(coalesce(col("penalty_amount"), lit(0))).alias("total_penalties_paid")
    )

    # -------------------------------
    # 🔗 Combine + Enrich
    # -------------------------------
    df = viol.join(insp, "address_clean", "left") \
             .join(district_map, "address_clean", "left") \
             .fillna(0)

    # -------------------------------
    # 🧠 Compliance Score (0–100)
    # -------------------------------
    df = df.withColumn(
        "compliance_score",
        round(
            (1 - (col("open_violations") / (col("total_violations") + 1))) * 50 +
            (col("inspection_pass_rate") * 50),
            2
        )
    )

    # -------------------------------
    # 📊 Compliance Categories
    # -------------------------------
    df = df.withColumn(
        "compliance_status",
        when(col("open_violations") == 0, "COMPLIANT")
        .when(col("open_violations") <= 2, "MINOR_ISSUES")
        .when(col("open_violations") <= 5, "AT_RISK")
        .otherwise("NON_COMPLIANT")
    )

    df = df.withColumn(
        "compliance_band",
        when(col("compliance_score") >= 80, "EXCELLENT")
        .when(col("compliance_score") >= 60, "GOOD")
        .when(col("compliance_score") >= 40, "POOR")
        .otherwise("CRITICAL")
    )

    # -------------------------------
    # 🚨 Repeat Offender Flag
    # -------------------------------
    df = df.withColumn(
        "repeat_offender_flag",
        when(col("total_violations") >= 3, 1).otherwise(0)
    )

    # -------------------------------
    # 🚀 Partitioned Window (FIXED)
    # -------------------------------
    window = Window.partitionBy("neighborhood_district") \
                   .orderBy(col("open_violations").desc(),
                            col("compliance_score").asc())

    df = df.withColumn(
        "inspection_priority_rank",
        dense_rank().over(window)
    )

    # -------------------------------
    # 💾 Write to Delta (FIXED)
    # -------------------------------
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sf_fire_gold.compliance_tracking")

    # -------------------------------
    # 📊 Debug / Demo Outputs
    # -------------------------------
        # -------------------------------
    # 📊 CLEAN DEMO OUTPUT
    # -------------------------------
    print("\n" + "="*70)
    print("⚠️ COMPLIANCE INTELLIGENCE DASHBOARD")
    print("="*70)

    # Summary
    total = df.count()
    compliant = df.filter(col("compliance_status") == "COMPLIANT").count()
    non_compliant = df.filter(col("compliance_status") == "NON_COMPLIANT").count()

    print("\n📊 OVERVIEW")
    print("-"*70)
    print(f"Total Properties        : {total}")
    print(f"Compliant Properties    : {compliant}")
    print(f"Non-Compliant Properties: {non_compliant}")

    # -------------------------------
    # 🚨 TOP NON-COMPLIANT
    # -------------------------------
    print("\n🚨 TOP 10 NON-COMPLIANT PROPERTIES")
    print("-"*70)

    df.filter(col("compliance_status") == "NON_COMPLIANT") \
      .orderBy(col("open_violations").desc()) \
      .select(
          "address_clean",
          "neighborhood_district",
          "open_violations",
          "total_violations",
          "compliance_score",
          "inspection_priority_rank"
      ).show(10, truncate=False)

    # -------------------------------
    # 🟢 TOP COMPLIANT
    # -------------------------------
    print("\n🟢 TOP 10 COMPLIANT PROPERTIES")
    print("-"*70)

    df.filter(col("compliance_status") == "COMPLIANT") \
      .orderBy(col("compliance_score").desc()) \
      .select(
          "address_clean",
          "neighborhood_district",
          "compliance_score",
          "inspection_pass_rate",
          "total_inspections"
      ).show(10, truncate=False)

    # -------------------------------
    # 📊 STATUS DISTRIBUTION
    # -------------------------------
    print("\n📊 COMPLIANCE STATUS DISTRIBUTION")
    print("-"*70)

    df.groupBy("compliance_status") \
      .count() \
      .orderBy("count", ascending=False) \
      .show()

    # -------------------------------
    # 🏷️ BAND DISTRIBUTION
    # -------------------------------
    print("\n📊 COMPLIANCE BAND DISTRIBUTION")
    print("-"*70)

    df.groupBy("compliance_band") \
      .count() \
      .orderBy("count", ascending=False) \
      .show()

    # -------------------------------
    # 🏆 PRIORITY INSPECTIONS
    # -------------------------------
    print("\n🏆 TOP 10 INSPECTION PRIORITY (PER DISTRICT)")
    print("-"*70)

    df.orderBy(col("inspection_priority_rank")) \
      .select(
          "address_clean",
          "neighborhood_district",
          "open_violations",
          "compliance_score",
          "inspection_priority_rank"
      ).show(10, truncate=False)

    print("\n" + "="*70)
    print("✅ COMPLIANCE ANALYSIS COMPLETE")
    print("="*70)


# ▶️ RUN
build_gold_compliance()

In [0]:
def build_gold_response_optimization():
    calls     = spark.table("sf_fire_silver.calls_for_service")
    incidents = spark.table("sf_fire_silver.incidents")

    # Response metrics by station and call type
    response = calls.groupBy("station_area", "battalion", "call_type").agg(
        count("call_number").alias("total_calls"),
        avg("dispatch_delay_seconds").alias("avg_dispatch_delay_secs"),
        avg("response_delay_seconds").alias("avg_response_delay_secs"),
        avg("call_duration_minutes").alias("avg_call_duration_mins"),
        max("dispatch_delay_seconds").alias("max_dispatch_delay_secs"),
        max("response_delay_seconds").alias("max_response_delay_secs")
    ).withColumn("avg_dispatch_delay_mins",
        round(col("avg_dispatch_delay_secs") / 60, 2)
    ).withColumn("avg_response_delay_mins",
        round(col("avg_response_delay_secs") / 60, 2)
    ).fillna(0)

    # Identify bottleneck reason
    response = response.withColumn("bottleneck_reason",
        when(
            col("avg_dispatch_delay_mins") > col("avg_response_delay_mins"),
            "DISPATCH DELAY — Slow unit assignment after call received"
        ).when(
            (col("avg_response_delay_mins") > 8) & (col("total_calls") > 100),
            "HIGH CALL VOLUME — Station overwhelmed by demand"
        ).when(
            col("avg_response_delay_mins") > 10,
            "TRAVEL TIME — Units too far from incident location"
        ).when(
            col("avg_dispatch_delay_mins") > 3,
            "PROCESSING DELAY — CAD entry or queuing lag"
        ).otherwise("NO SIGNIFICANT BOTTLENECK")
    )

    # Performance tier
    response = response.withColumn("performance_tier",
        when(col("avg_response_delay_mins") <= 4, "🟢 EXCELLENT")
        .when(col("avg_response_delay_mins") <= 6, "🟡 ACCEPTABLE")
        .when(col("avg_response_delay_mins") <= 10, "🟠 NEEDS IMPROVEMENT")
        .otherwise("🔴 CRITICAL")
    )

    # Improvement suggestion
    response = response.withColumn("improvement_suggestion",
        when(
            col("bottleneck_reason").contains("DISPATCH"),
            "Automate unit pre-assignment for recurring call types in this area"
        ).when(
            col("bottleneck_reason").contains("HIGH CALL VOLUME"),
            "Add supplementary unit or mutual aid agreement for peak hours"
        ).when(
            col("bottleneck_reason").contains("TRAVEL TIME"),
            "Reposition nearest unit closer to this station coverage zone"
        ).when(
            col("bottleneck_reason").contains("PROCESSING"),
            "Review CAD workflow — reduce manual entry steps at dispatch"
        ).otherwise("Maintain current deployment strategy")
    )

    response.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sf_fire_gold.response_optimization")

    print("=" * 60)
    print("🚒 RESPONSE OPTIMIZATION")
    print("=" * 60)

    print("\n📊 Performance Tier Breakdown:")
    response.groupBy("performance_tier").count() \
        .orderBy("count", ascending=False).show(truncate=False)

    print("\n🔴 Top 10 Slowest Station + Call Type Combos:")
    response.filter(col("performance_tier") == "🔴 CRITICAL") \
        .orderBy(col("avg_response_delay_mins").desc()) \
        .select(
            "station_area", "call_type", "total_calls",
            "avg_dispatch_delay_mins", "avg_response_delay_mins",
            "bottleneck_reason", "improvement_suggestion"
        ).show(10, truncate=False)

build_gold_response_optimization()

In [0]:
def build_gold_inspection_effectiveness():
    inspections = spark.table("sf_fire_silver.inspections")
    violations  = spark.table("sf_fire_silver.violations")

    # ── Inspection outcomes per property ────────────────
    insp_agg = inspections.groupBy(
        "address_clean", "neighborhood_district", "battalion"
    ).agg(
        count("inspection_number").alias("total_inspections"),
        sum("passed_flag").alias("passed_inspections"),
        sum("has_violation").alias("failed_inspections"),
        avg("passed_flag").alias("pass_rate"),
        avg("inspection_duration_days").alias("avg_inspection_duration_days"),
        sum(coalesce(col("penalty_amount"), lit(0))).alias("total_penalties"),
        sum(coalesce(col("invoice_amount"), lit(0))).alias("total_invoice_amount"),
        count(when(col("inspection_type") == "01", 1)).alias("initial_inspections"),
        count(when(col("billable_inspection") == True, 1)).alias("billable_inspections")
    ).withColumn("fail_rate",
        round(col("failed_inspections") / (col("total_inspections") + 1) * 100, 2)
    ).withColumn("pass_rate_pct",
        round(col("pass_rate") * 100, 2)
    )

    # ── Violation resolution linked to inspections ───────
    viol_agg = violations.groupBy("address_clean").agg(
        count("violation_number").alias("violations_ever_issued"),
        sum("is_open").alias("still_open_after_inspection"),
        avg("days_to_close").alias("avg_days_violation_open"),
        sum(when(col("status") == "Closed", 1).otherwise(0)).alias("violations_resolved")
    ).withColumn("violation_closure_rate",
        round(col("violations_resolved") / (col("violations_ever_issued") + 1) * 100, 2)
    )

    # ── Join ─────────────────────────────────────────────
    df = insp_agg.join(viol_agg, "address_clean", "left").fillna(0)

    # ── Effectiveness Score 0-100 ────────────────────────
    df = df.withColumn("effectiveness_score",
        round(
            col("pass_rate_pct") * 0.35 +
            col("violation_closure_rate") * 0.35 +
            (100 - col("fail_rate")) * 0.20 +
            when(col("avg_days_violation_open") < 30, 10)
            .when(col("avg_days_violation_open") < 90, 5)
            .otherwise(0),
            2
        )
    )

    # ── Effectiveness Band ───────────────────────────────
    df = df.withColumn("effectiveness_band",
        when(col("effectiveness_score") >= 80, "🟢 HIGHLY EFFECTIVE")
        .when(col("effectiveness_score") >= 60, "🟡 MODERATELY EFFECTIVE")
        .when(col("effectiveness_score") >= 40, "🟠 LOW EFFECTIVENESS")
        .otherwise("🔴 INEFFECTIVE")
    )

    # ── Re-inspection Risk Flag ──────────────────────────
    df = df.withColumn("reinspection_needed",
        when(
            (col("still_open_after_inspection") > 0) &
            (col("avg_days_violation_open") > 90),
            "🔴 URGENT — violations open > 90 days"
        ).when(
            col("fail_rate") > 50,
            "🟠 HIGH — majority of inspections failed"
        ).when(
            col("violations_resolved") < col("failed_inspections"),
            "🟡 MEDIUM — more failures than resolutions"
        ).otherwise("🟢 NOT REQUIRED")
    )

    # ── Inspector Workload by District ───────────────────
    district_effectiveness = df.groupBy(
        "neighborhood_district", "battalion"
    ).agg(
        count("address_clean").alias("properties_inspected"),
        avg("effectiveness_score").alias("avg_effectiveness_score"),
        avg("pass_rate_pct").alias("avg_pass_rate"),
        avg("violation_closure_rate").alias("avg_closure_rate"),
        sum("total_penalties").alias("total_penalties_collected"),
        sum("still_open_after_inspection").alias("total_open_violations"),
        avg("avg_days_violation_open").alias("avg_days_violations_open")
    ).withColumn("district_effectiveness_tier",
        when(col("avg_effectiveness_score") >= 80, "🟢 HIGH")
        .when(col("avg_effectiveness_score") >= 60, "🟡 MEDIUM")
        .otherwise("🔴 LOW")
    )

    # ── Save both tables ─────────────────────────────────
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sf_fire_gold.inspection_effectiveness")

    district_effectiveness.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sf_fire_gold.inspection_effectiveness_by_district")

    # ── Output ───────────────────────────────────────────
    print("=" * 60)
    print("🔍 INSPECTION EFFECTIVENESS")
    print("=" * 60)
    print(f"\nTotal Properties Inspected : {df.count()}")

    print("\n📊 Effectiveness Band Distribution:")
    df.groupBy("effectiveness_band").count() \
        .orderBy("count", ascending=False).show(truncate=False)

    print("\n🚩 Re-inspection Flag Distribution:")
    df.groupBy("reinspection_needed").count() \
        .orderBy("count", ascending=False).show(truncate=False)

    print("\n🏘️ District Effectiveness Breakdown:")
    district_effectiveness.orderBy(
        col("avg_effectiveness_score").asc()
    ).select(
        "neighborhood_district",
        "properties_inspected",
        "avg_effectiveness_score",
        "avg_pass_rate",
        "avg_closure_rate",
        "total_open_violations",
        "district_effectiveness_tier"
    ).show(15, truncate=False)

    print("\n🔴 Top 10 Properties Needing Urgent Re-inspection:")
    df.filter(col("reinspection_needed").contains("URGENT")) \
        .orderBy(col("avg_days_violation_open").desc()) \
        .select(
            "address_clean",
            "total_inspections",
            "failed_inspections",
            "still_open_after_inspection",
            "avg_days_violation_open",
            "effectiveness_score",
            "reinspection_needed"
        ).show(10, truncate=False)

build_gold_inspection_effectiveness()

In [0]:
# Check actual values in these columns
print("=== inspection_status values ===")
spark.table("sf_fire_silver.inspections") \
    .groupBy("inspection_status").count() \
    .orderBy("count", ascending=False).show(20, truncate=False)

print("=== violation status values ===")
spark.table("sf_fire_silver.violations") \
    .groupBy("status").count() \
    .orderBy("count", ascending=False).show(20, truncate=False)

print("=== passed_flag distribution ===")
spark.table("sf_fire_silver.inspections") \
    .groupBy("passed_flag").count().show()

print("=== has_violation distribution ===")
spark.table("sf_fire_silver.inspections") \
    .groupBy("has_violation").count().show()

In [0]:
from pyspark.sql.functions import col, when

# Fix passed_flag in silver inspections
df = spark.table("sf_fire_silver.inspections")

df_fixed = df.withColumn("passed_flag",
    when(col("inspection_status") == "Completed", 1).otherwise(0)
).withColumn("needs_followup",
    when(col("inspection_status") == "Open/Follow-Up Needed", 1).otherwise(0)
).withColumn("is_pending",
    when(col("inspection_status") == "Pending", 1).otherwise(0)
)

df_fixed.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("sf_fire_silver.inspections")

print("✅ Silver inspections fixed")
print("passed_flag distribution:")
df_fixed.groupBy("passed_flag").count().show()

In [0]:
# Fix is_open in silver violations — abated/closed/rescinded = resolved
df = spark.table("sf_fire_silver.violations")

df_fixed = df.withColumn("is_open",
    when(col("status") == "open", 1)
    .when(col("status") == "order to abate", 1)
    .when(col("status") == "referred to hearing", 1)
    .when(col("status") == "city attorney referral", 1)
    .otherwise(0)
).withColumn("is_resolved",
    when(col("status") == "abated", 1)
    .when(col("status") == "closed", 1)
    .when(col("status") == "rescinded", 1)
    .otherwise(0)
)

df_fixed.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("sf_fire_silver.violations")

print("✅ Silver violations fixed")
print("is_open distribution:")
df_fixed.groupBy("status", "is_open", "is_resolved").count() \
    .orderBy("count", ascending=False).show(truncate=False)

In [0]:
from pyspark.sql.functions import col, count, sum, avg, min, max, when, coalesce, lit, round

def rebuild_gold_property_table():
    incidents   = spark.table("sf_fire_silver.incidents")
    violations  = spark.table("sf_fire_silver.violations")
    inspections = spark.table("sf_fire_silver.inspections")
    permits     = spark.table("sf_fire_silver.permits")

    viol_agg = violations.groupBy("address_clean").agg(
        count("violation_number").alias("total_violations"),
        sum("is_open").alias("open_violations"),
        sum("is_resolved").alias("resolved_violations")
    ).withColumn("open_violation_ratio",
        col("open_violations") / (col("total_violations") + 1))

    inc_agg = incidents.groupBy("address_clean").agg(
        count("incident_number").alias("incident_history_count"),
        sum("estimated_property_loss").alias("total_property_loss"),
        avg("response_time_minutes").alias("avg_response_time"),
        sum(coalesce(col("fire_fatalities"), lit(0))).alias("total_fire_fatalities")
    )

    insp_agg = inspections.groupBy("address_clean").agg(
        count("inspection_number").alias("total_inspections"),
        avg("passed_flag").alias("pass_rate"),
        sum("has_violation").alias("inspections_with_violations"),
        sum("needs_followup").alias("followup_needed_count")
    )

    perm_agg = permits.groupBy("address_clean").agg(
        count("permit_number").alias("total_permits"),
        sum("is_expired").alias("expired_permits")
    ).withColumn("expired_permit_ratio",
        col("expired_permits") / (col("total_permits") + 1))

    gold = viol_agg \
        .join(inc_agg,   "address_clean", "left") \
        .join(insp_agg,  "address_clean", "left") \
        .join(perm_agg,  "address_clean", "left") \
        .fillna(0)

    gold = gold.withColumn("risk_score",
        round(
            col("open_violation_ratio") * 30 +
            col("incident_history_count") * 10 +
            col("expired_permit_ratio") * 20 +
            (1 - col("pass_rate")) * 20 +
            col("inspections_with_violations") * 5 +
            col("total_fire_fatalities") * 15,
            2
        )
    ).withColumn("risk_tier",
        when(col("risk_score") >= 60, "HIGH")
        .when(col("risk_score") >= 30, "MEDIUM")
        .otherwise("LOW")
    ).withColumn("high_risk_label",
        when(col("risk_tier") == "HIGH", 1).otherwise(0)
    )

    gold.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sf_fire_gold.property_risk_master")

    print("✅ Gold property_risk_master rebuilt")
    gold.groupBy("risk_tier").count().show()

rebuild_gold_property_table()

In [0]:
def rebuild_inspection_effectiveness():
    inspections = spark.table("sf_fire_silver.inspections")
    violations  = spark.table("sf_fire_silver.violations")

    insp_agg = inspections.groupBy(
        "address_clean", "neighborhood_district", "battalion"
    ).agg(
        count("inspection_number").alias("total_inspections"),
        sum("passed_flag").alias("passed_inspections"),
        sum("has_violation").alias("failed_inspections"),
        sum("needs_followup").alias("followup_needed_count"),
        avg("passed_flag").alias("pass_rate"),
        avg("inspection_duration_days").alias("avg_inspection_duration_days"),
        sum(coalesce(col("penalty_amount"), lit(0))).alias("total_penalties"),
        sum(coalesce(col("invoice_amount"), lit(0))).alias("total_invoice_amount")
    ).withColumn("fail_rate",
        round(col("failed_inspections") / (col("total_inspections") + 1) * 100, 2)
    ).withColumn("pass_rate_pct",
        round(col("pass_rate") * 100, 2)
    )

    viol_agg = violations.groupBy("address_clean").agg(
        count("violation_number").alias("violations_ever_issued"),
        sum("is_open").alias("still_open_after_inspection"),
        sum("is_resolved").alias("violations_resolved"),
        avg("days_to_close").alias("avg_days_violation_open")
    ).withColumn("violation_closure_rate",
        round(col("violations_resolved") / (col("violations_ever_issued") + 1) * 100, 2)
    )

    df = insp_agg.join(viol_agg, "address_clean", "left").fillna(0)

    df = df.withColumn("effectiveness_score",
        round(
            col("pass_rate_pct") * 0.35 +
            col("violation_closure_rate") * 0.35 +
            (100 - col("fail_rate")) * 0.20 +
            when(col("avg_days_violation_open") < 30, 10)
            .when(col("avg_days_violation_open") < 90, 5)
            .otherwise(0),
            2
        )
    ).withColumn("effectiveness_band",
        when(col("effectiveness_score") >= 80, "🟢 HIGHLY EFFECTIVE")
        .when(col("effectiveness_score") >= 60, "🟡 MODERATELY EFFECTIVE")
        .when(col("effectiveness_score") >= 40, "🟠 LOW EFFECTIVENESS")
        .otherwise("🔴 INEFFECTIVE")
    ).withColumn("reinspection_needed",
        when(
            (col("still_open_after_inspection") > 0) &
            (col("avg_days_violation_open") > 90),
            "🔴 URGENT — violations open > 90 days"
        ).when(
            col("fail_rate") > 50,
            "🟠 HIGH — majority of inspections failed"
        ).when(
            col("violations_resolved") < col("failed_inspections"),
            "🟡 MEDIUM — more failures than resolutions"
        ).otherwise("🟢 NOT REQUIRED")
    )

    district_df = df.groupBy("neighborhood_district", "battalion").agg(
        count("address_clean").alias("properties_inspected"),
        avg("effectiveness_score").alias("avg_effectiveness_score"),
        avg("pass_rate_pct").alias("avg_pass_rate"),
        avg("violation_closure_rate").alias("avg_closure_rate"),
        sum("total_penalties").alias("total_penalties_collected"),
        sum("still_open_after_inspection").alias("total_open_violations"),
        avg("avg_days_violation_open").alias("avg_days_violations_open")
    ).withColumn("district_effectiveness_tier",
        when(col("avg_effectiveness_score") >= 80, "🟢 HIGH")
        .when(col("avg_effectiveness_score") >= 60, "🟡 MEDIUM")
        .otherwise("🔴 LOW")
    )

    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sf_fire_gold.inspection_effectiveness")

    district_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sf_fire_gold.inspection_effectiveness_by_district")

    print("=" * 60)
    print("🔍 INSPECTION EFFECTIVENESS — REBUILT")
    print("=" * 60)
    print(f"\nTotal Properties: {df.count()}")

    print("\n📊 Effectiveness Band Distribution:")
    df.groupBy("effectiveness_band").count() \
        .orderBy("count", ascending=False).show(truncate=False)

    print("\n🚩 Re-inspection Flag Distribution:")
    df.groupBy("reinspection_needed").count() \
        .orderBy("count", ascending=False).show(truncate=False)

    print("\n🔴 Top 10 Urgent Re-inspection Properties:")
    df.filter(col("reinspection_needed").contains("URGENT")) \
        .orderBy(col("avg_days_violation_open").desc()) \
        .select(
            "address_clean", "total_inspections", "failed_inspections",
            "still_open_after_inspection", "avg_days_violation_open",
            "violation_closure_rate", "effectiveness_score"
        ).show(10, truncate=False)

rebuild_inspection_effectiveness()

In [0]:
def rebuild_compliance():
    violations  = spark.table("sf_fire_silver.violations")
    inspections = spark.table("sf_fire_silver.inspections")

    viol = violations.groupBy("address_clean", "neighborhood_district").agg(
        count("violation_number").alias("total_violations"),
        sum("is_open").alias("open_violations"),
        sum("is_resolved").alias("closed_violations"),
        avg("days_to_close").alias("avg_days_to_resolve"),
        avg("days_since_violation").alias("avg_age_of_violations"),
        max("days_since_violation").alias("oldest_violation_days")
    ).withColumn("resolution_rate",
        round(col("closed_violations") / (col("total_violations") + 1) * 100, 2)
    )

    insp = inspections.groupBy("address_clean").agg(
        count("inspection_number").alias("total_inspections"),
        avg("passed_flag").alias("inspection_pass_rate"),
        sum("has_violation").alias("failed_inspections"),
        sum("needs_followup").alias("followup_count"),
        sum(coalesce(col("penalty_amount"), lit(0))).alias("total_penalties")
    )

    compliance = viol.join(insp, "address_clean", "left").fillna(0)

    compliance = compliance.withColumn("compliance_band",
        when(
            (col("open_violations") == 0) & (col("inspection_pass_rate") >= 0.8),
            "✅ COMPLIANT"
        ).when(
            (col("open_violations") <= 2) & (col("resolution_rate") >= 50),
            "🟡 MINOR ISSUES"
        ).when(
            (col("open_violations") <= 5) | (col("inspection_pass_rate") < 0.5),
            "🟠 AT RISK"
        ).otherwise("🔴 NON COMPLIANT")
    ).withColumn("urgency_flag",
        when(
            (col("open_violations") >= 5) & (col("avg_age_of_violations") > 365),
            "CRITICAL — Long standing violations"
        ).when(
            col("total_penalties") > 5000,
            "HIGH — Significant penalties accumulated"
        ).when(
            col("resolution_rate") < 25,
            "HIGH — Very low resolution rate"
        ).when(
            col("open_violations") >= 3,
            "MEDIUM — Multiple open violations"
        ).otherwise("LOW")
    )

    compliance.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sf_fire_gold.compliance_tracking")

    print("=" * 60)
    print("📋 COMPLIANCE TRACKING — REBUILT")
    print("=" * 60)
    print(f"\nTotal Properties: {compliance.count()}")

    print("\n📊 Compliance Band Distribution:")
    compliance.groupBy("compliance_band").count() \
        .orderBy("count", ascending=False).show(truncate=False)

    print("\n⚠️ Urgency Flag Distribution:")
    compliance.groupBy("urgency_flag").count() \
        .orderBy("count", ascending=False).show(truncate=False)

    print("\n🔴 Top 10 Non-Compliant Properties:")
    compliance.filter(col("compliance_band") == "🔴 NON COMPLIANT") \
        .orderBy(col("open_violations").desc()) \
        .select(
            "address_clean", "open_violations", "total_violations",
            "resolution_rate", "avg_days_to_resolve", "urgency_flag"
        ).show(10, truncate=False)

rebuild_compliance()

In [0]:
def rebuild_district_summary():
    gold       = spark.table("sf_fire_gold.property_risk_master")
    violations = spark.table("sf_fire_silver.violations")
    incidents  = spark.table("sf_fire_silver.incidents")

    viol_district = violations.select(
        "address_clean", "neighborhood_district", "battalion"
    ).dropDuplicates(["address_clean"])

    inc_district = incidents.select(
        "address_clean", "neighborhood_district", "battalion"
    ).dropDuplicates(["address_clean"])

    district_info = viol_district.unionByName(inc_district) \
        .dropDuplicates(["address_clean"])

    df = gold.join(district_info, "address_clean", "left")

    district = df.groupBy("neighborhood_district", "battalion").agg(
        count("address_clean").alias("total_properties"),
        sum("high_risk_label").alias("high_risk_count"),
        avg("risk_score").alias("avg_risk_score"),
        max("risk_score").alias("max_risk_score"),
        sum("total_violations").alias("total_violations"),
        sum("open_violations").alias("open_violations"),
        sum("incident_history_count").alias("total_incidents"),
        sum("total_fire_fatalities").alias("total_fatalities"),
        sum("total_property_loss").alias("total_property_loss"),
        avg("avg_response_time").alias("avg_response_time_mins"),
        sum("expired_permits").alias("total_expired_permits"),
        sum("inspections_with_violations").alias("total_failed_inspections")
    ).withColumn("high_risk_pct",
        round(col("high_risk_count") / (col("total_properties") + 1) * 100, 2)
    ).withColumn("open_violation_rate",
        round(col("open_violations") / (col("total_violations") + 1) * 100, 2)
    ).withColumn("district_priority_score",
        round(
            col("avg_risk_score") * 0.3 +
            col("high_risk_pct") * 0.25 +
            col("open_violation_rate") * 0.2 +
            col("total_fatalities") * 10 +
            col("avg_response_time_mins") * 0.15,
            2
        )
    )

    from pyspark.sql.window import Window
    from pyspark.sql.functions import dense_rank
    window = Window.orderBy(col("district_priority_score").desc())

    district = district \
        .withColumn("priority_rank", dense_rank().over(window)) \
        .withColumn("priority_action",
            when(col("priority_rank") <= 3,  "🔴 IMMEDIATE ACTION")
            .when(col("priority_rank") <= 7,  "🟠 HIGH PRIORITY")
            .when(col("priority_rank") <= 12, "🟡 MONITOR CLOSELY")
            .otherwise("🟢 ROUTINE")
        ).withColumn("recommended_action",
            when(col("total_fatalities") > 0,
                "Deploy additional units — fatalities recorded")
            .when(col("open_violation_rate") > 50,
                "Enforce violation clearance — >50% unresolved")
            .when(col("avg_response_time_mins") > 10,
                "Reposition units — response time exceeds 10 mins")
            .when(col("high_risk_pct") > 20,
                "Conduct targeted inspections — high risk density")
            .when(col("total_expired_permits") > 10,
                "Permit renewal campaign needed")
            .otherwise("Maintain current protocols")
        )

    district.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sf_fire_gold.district_risk_summary")

    print("=" * 60)
    print("📍 DISTRICT RISK SUMMARY — REBUILT")
    print("=" * 60)
    print(f"\nTotal Districts: {district.count()}")

    print("\n🔴 Top 10 Priority Districts:")
    district.select(
        "neighborhood_district", "priority_rank", "priority_action",
        "avg_risk_score", "high_risk_pct", "recommended_action"
    ).orderBy("priority_rank").show(10, truncate=False)

rebuild_district_summary()

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import col
import pandas as pd

# ── 1. Load Gold Table ───────────────────────────────────
df = spark.table("sf_fire_gold.property_risk_master")

# ── 2. Feature Columns ──────────────────────────────────
feature_cols = [
    "open_violations",
    "total_violations",
    "open_violation_ratio",
    "incident_history_count",
    "total_property_loss",
    "avg_response_time",
    "total_fire_fatalities",
    "inspections_with_violations",
    "pass_rate",
    "expired_permit_ratio",
    "total_permits"
]

# ── 3. Prepare Data ──────────────────────────────────────
df_ml = df.select(feature_cols + ["high_risk_label"]) \
          .fillna(0) \
          .withColumnRenamed("high_risk_label", "label")

# Train / Test Split — 80/20
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows : {train_df.count()}")
print(f"Testing rows  : {test_df.count()}")
print(f"High Risk in Train: {train_df.filter(col('label')==1).count()}")
print(f"Low Risk in Train : {train_df.filter(col('label')==0).count()}")

# ── 4. Assemble Features ─────────────────────────────────
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

# ── 5. Random Forest ─────────────────────────────────────
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    seed=42
)

# ── 6. Logistic Regression ───────────────────────────────
lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=100
)

# ── 7. Build Pipelines ───────────────────────────────────
rf_pipeline = Pipeline(stages=[assembler, rf])
lr_pipeline = Pipeline(stages=[assembler, lr])

# ── 8. Train Both Models ─────────────────────────────────
print("\nTraining Random Forest...")
rf_model = rf_pipeline.fit(train_df)
print("✅ Random Forest trained")

print("Training Logistic Regression...")
lr_model = lr_pipeline.fit(train_df)
print("✅ Logistic Regression trained")

# ── 9. Predictions ───────────────────────────────────────
rf_preds = rf_model.transform(test_df)
lr_preds = lr_model.transform(test_df)

# ── 10. Evaluators ───────────────────────────────────────
binary_eval   = BinaryClassificationEvaluator(labelCol="label")
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy")
precision_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
recall_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall")
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1")

# ── 11. Scores ───────────────────────────────────────────
rf_auc       = binary_eval.evaluate(rf_preds)
rf_accuracy  = accuracy_eval.evaluate(rf_preds)
rf_precision = precision_eval.evaluate(rf_preds)
rf_recall    = recall_eval.evaluate(rf_preds)
rf_f1        = f1_eval.evaluate(rf_preds)

lr_auc       = binary_eval.evaluate(lr_preds)
lr_accuracy  = accuracy_eval.evaluate(lr_preds)
lr_precision = precision_eval.evaluate(lr_preds)
lr_recall    = recall_eval.evaluate(lr_preds)
lr_f1        = f1_eval.evaluate(lr_preds)

# ── 12. Results Table ────────────────────────────────────
print("\n" + "="*60)
print("📊 MODEL COMPARISON RESULTS")
print("="*60)
print(f"{'Metric':<20} {'Random Forest':>15} {'Logistic Reg':>15}")
print("-"*60)
print(f"{'AUC-ROC':<20} {rf_auc:>15.4f} {lr_auc:>15.4f}")
print(f"{'Accuracy':<20} {rf_accuracy:>15.4f} {lr_accuracy:>15.4f}")
print(f"{'Precision':<20} {rf_precision:>15.4f} {lr_precision:>15.4f}")
print(f"{'Recall':<20} {rf_recall:>15.4f} {lr_recall:>15.4f}")
print(f"{'F1 Score':<20} {rf_f1:>15.4f} {lr_f1:>15.4f}")
print("="*60)

# ── 13. Feature Importances ──────────────────────────────
print("\n🌲 TOP FEATURES DRIVING FIRE RISK (Random Forest):")
print("-"*45)
rf_feature_importance = rf_model.stages[-1].featureImportances
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_feature_importance.toArray()
}).sort_values("importance", ascending=False)

for _, row in importance_df.iterrows():
    bar = "█" * int(row["importance"] * 100)
    print(f"{row['feature']:<35} {row['importance']:.4f}  {bar}")

# ── 14. Confusion Matrix ─────────────────────────────────
print("\n📊 CONFUSION MATRIX — Random Forest:")
print("-"*40)
rf_preds.groupBy("label", "prediction").count() \
    .orderBy("label", "prediction").show()

# ── 15. Save Predictions to Gold ─────────────────────────
gold = spark.table("sf_fire_gold.property_risk_master")
assembler_pred = VectorAssembler(
    inputCols=feature_cols, outputCol="features", handleInvalid="skip"
)
full_df = assembler_pred.transform(gold.fillna(0))
final_predictions = rf_model.stages[-1].transform(full_df)

final_output = final_predictions.select(
    "address_clean",
    "risk_score",
    "risk_tier",
    "high_risk_label",
    col("prediction").alias("ml_prediction"),
    col("probability").alias("ml_probability")
)

final_output.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("sf_fire_gold.ml_risk_predictions")

print("\n✅ ML predictions saved to sf_fire_gold.ml_risk_predictions")
print(f"\nTotal properties scored : {final_output.count()}")
print("\nML Prediction Distribution:")
final_output.groupBy("ml_prediction").count().show()
print("\nRisk Tier vs ML Prediction Agreement:")
final_output.groupBy("risk_tier", "ml_prediction").count() \
    .orderBy("risk_tier", "ml_prediction").show()

In [0]:
from pyspark.sql.functions import col, when, round, coalesce, lit, count, sum, avg

def rebuild_gold_three_tier():
    df = spark.table("sf_fire_gold.property_risk_master")

    df_updated = df.withColumn("risk_label",
        when(col("risk_score") >= 60, 2)   # CRITICAL
        .when(col("risk_score") >= 30, 1)   # HIGH
        .otherwise(0)                        # NO RISK
    ).withColumn("risk_category",
        when(col("risk_score") >= 60, "CRITICAL")
        .when(col("risk_score") >= 30, "HIGH")
        .otherwise("NO RISK")
    )

    df_updated.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sf_fire_gold.property_risk_master")

    print("✅ Gold table updated with 3-tier labels")
    print("\n📊 Risk Category Distribution:")
    df_updated.groupBy("risk_category", "risk_label") \
        .count() \
        .orderBy("risk_label").show()

rebuild_gold_three_tier()

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, when
import pandas as pd

# ── 1. Load Data ─────────────────────────────────────────
df = spark.table("sf_fire_gold.property_risk_master")

feature_cols = [
    "open_violations",
    "total_violations",
    "open_violation_ratio",
    "incident_history_count",
    "total_property_loss",
    "avg_response_time",
    "total_fire_fatalities",
    "inspections_with_violations",
    "pass_rate",
    "expired_permit_ratio",
    "total_permits"
]

# ── 2. Prepare ───────────────────────────────────────────
df_ml = df.select(feature_cols + ["risk_label", "risk_category"]) \
          .fillna(0) \
          .withColumnRenamed("risk_label", "label")

print("📊 Original Label Distribution:")
df_ml.groupBy("label", "risk_category").count().orderBy("label").show()

# ── 3. Balance Classes ───────────────────────────────────
no_risk  = df_ml.filter(col("label") == 0)
high     = df_ml.filter(col("label") == 1)
critical = df_ml.filter(col("label") == 2)

high_ratio     = int(no_risk.count() / high.count())
critical_ratio = int(no_risk.count() / critical.count())

high_oversampled     = high.sample(withReplacement=True, fraction=float(high_ratio), seed=42)
critical_oversampled = critical.sample(withReplacement=True, fraction=float(critical_ratio), seed=42)

balanced_df = no_risk.union(high_oversampled).union(critical_oversampled)

print("📊 Balanced Label Distribution:")
balanced_df.groupBy("label", "risk_category").count().orderBy("label").show()

# ── 4. Train / Test Split ────────────────────────────────
train_df, test_df = balanced_df.randomSplit([0.8, 0.2], seed=42)
print(f"Training rows : {train_df.count()}")
print(f"Testing rows  : {test_df.count()}")

# ── 5. Assembler ─────────────────────────────────────────
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

# ── 6. Models ────────────────────────────────────────────
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    seed=42
)

lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=100,
    family="multinomial"
)

# ── 7. Pipelines ─────────────────────────────────────────
rf_pipeline = Pipeline(stages=[assembler, rf])
lr_pipeline = Pipeline(stages=[assembler, lr])

# ── 8. Train ─────────────────────────────────────────────
print("\nTraining Random Forest...")
rf_model = rf_pipeline.fit(train_df)
print("✅ Random Forest trained")

print("Training Logistic Regression...")
lr_model = lr_pipeline.fit(train_df)
print("✅ Logistic Regression trained")

# ── 9. Evaluate on real unbalanced test set ──────────────
_, real_test = df_ml.randomSplit([0.8, 0.2], seed=42)

rf_preds = rf_model.transform(real_test.fillna(0))
lr_preds = lr_model.transform(real_test.fillna(0))

def evaluate_model(preds, name):
    metrics = ["accuracy", "weightedPrecision", "weightedRecall", "f1"]
    results = {}
    for metric in metrics:
        evaluator = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName=metric
        )
        results[metric] = evaluator.evaluate(preds)
    return results

rf_results = evaluate_model(rf_preds, "Random Forest")
lr_results = evaluate_model(lr_preds, "Logistic Regression")

# ── 10. Results ──────────────────────────────────────────
print("\n" + "="*60)
print("📊 MODEL COMPARISON — 3 CLASS RISK PREDICTION")
print("="*60)
print(f"{'Metric':<20} {'Random Forest':>15} {'Logistic Reg':>15}")
print("-"*60)
print(f"{'Accuracy':<20} {rf_results['accuracy']:>15.4f} {lr_results['accuracy']:>15.4f}")
print(f"{'Precision':<20} {rf_results['weightedPrecision']:>15.4f} {lr_results['weightedPrecision']:>15.4f}")
print(f"{'Recall':<20} {rf_results['weightedRecall']:>15.4f} {lr_results['weightedRecall']:>15.4f}")
print(f"{'F1 Score':<20} {rf_results['f1']:>15.4f} {lr_results['f1']:>15.4f}")
print("="*60)

# ── 11. Confusion Matrix ─────────────────────────────────
print("\n📊 CONFUSION MATRIX — Random Forest:")
print("(0=NO RISK  1=HIGH  2=CRITICAL)")
print("-"*40)
rf_preds.groupBy("label", "prediction").count() \
    .orderBy("label", "prediction").show()

# ── 12. Feature Importances ──────────────────────────────
print("\n🌲 TOP FEATURES DRIVING FIRE RISK:")
print("-"*50)
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.stages[-1].featureImportances.toArray()
}).sort_values("importance", ascending=False)

for _, row in importance_df.iterrows():
    bar = "█" * int(row["importance"] * 100)
    print(f"{row['feature']:<35} {row['importance']:.4f}  {bar}")

# ── 13. Save Predictions ─────────────────────────────────
# Drop features column if already exists before assembling
gold = spark.table("sf_fire_gold.property_risk_master").fillna(0)

if "features" in gold.columns:
    gold = gold.drop("features")

final_preds = rf_model.transform(gold)

final_output = final_preds.select(
    "address_clean",
    "risk_score",
    "risk_tier",
    "risk_category",
    "risk_label",
    col("prediction").alias("ml_prediction"),
    col("probability").alias("ml_probability")
).withColumn("ml_risk_category",
    when(col("ml_prediction") == 2, "CRITICAL")
    .when(col("ml_prediction") == 1, "HIGH")
    .otherwise("NO RISK")
)

final_output.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("sf_fire_gold.ml_risk_predictions")

print("\n✅ Saved to sf_fire_gold.ml_risk_predictions")
print(f"Total Properties Scored: {final_output.count()}")

print("\n📊 ML Prediction Distribution:")
final_output.groupBy("ml_risk_category").count() \
    .orderBy("ml_risk_category").show()

print("\n📊 Actual vs Predicted Agreement:")
final_output.groupBy("risk_category", "ml_risk_category").count() \
    .orderBy("risk_category", "ml_risk_category").show()

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline, PipelineModel
from pyspark.sql.functions import col, when
import pandas as pd

# ── 1. Load Data ─────────────────────────────────────────
df = spark.table("sf_fire_gold.property_risk_master")

feature_cols = [
    "open_violations",
    "total_violations",
    "open_violation_ratio",
    "incident_history_count",
    "total_property_loss",
    "avg_response_time",
    "total_fire_fatalities",
    "inspections_with_violations",
    "pass_rate",
    "expired_permit_ratio",
    "total_permits"
]

# ── 2. Prepare ───────────────────────────────────────────
df_ml = df.select(feature_cols + ["risk_label", "risk_category"]) \
          .fillna(0) \
          .withColumnRenamed("risk_label", "label")

# ── 3. Balance Classes ───────────────────────────────────
no_risk  = df_ml.filter(col("label") == 0)
high     = df_ml.filter(col("label") == 1)
critical = df_ml.filter(col("label") == 2)

high_ratio     = int(no_risk.count() / high.count())
critical_ratio = int(no_risk.count() / critical.count())

high_oversampled     = high.sample(withReplacement=True, fraction=float(high_ratio), seed=42)
critical_oversampled = critical.sample(withReplacement=True, fraction=float(critical_ratio), seed=42)

balanced_df = no_risk.union(high_oversampled).union(critical_oversampled)

print("📊 Balanced Label Distribution:")
balanced_df.groupBy("label", "risk_category").count().orderBy("label").show()

# ── 4. Train / Test Split ────────────────────────────────
train_df, test_df = balanced_df.randomSplit([0.8, 0.2], seed=42)
print(f"Training rows : {train_df.count()}")
print(f"Testing rows  : {test_df.count()}")

# ── 5. Assembler + Models ────────────────────────────────
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    seed=42
)

lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=100,
    family="multinomial"
)

rf_pipeline = Pipeline(stages=[assembler, rf])
lr_pipeline = Pipeline(stages=[assembler, lr])

# ── 6. Train ─────────────────────────────────────────────
print("\nTraining Random Forest...")
rf_model = rf_pipeline.fit(train_df)
print("✅ Random Forest trained")

print("Training Logistic Regression...")
lr_model = lr_pipeline.fit(train_df)
print("✅ Logistic Regression trained")

# ── 7. Evaluate ──────────────────────────────────────────
_, real_test = df_ml.randomSplit([0.8, 0.2], seed=42)
rf_preds = rf_model.transform(real_test.fillna(0))
lr_preds = lr_model.transform(real_test.fillna(0))

def evaluate_model(preds):
    results = {}
    for metric in ["accuracy", "weightedPrecision", "weightedRecall", "f1"]:
        evaluator = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName=metric
        )
        results[metric] = evaluator.evaluate(preds)
    return results

rf_results = evaluate_model(rf_preds)
lr_results = evaluate_model(lr_preds)

print("\n" + "="*60)
print("📊 MODEL COMPARISON — 3 CLASS RISK PREDICTION")
print("="*60)
print(f"{'Metric':<20} {'Random Forest':>15} {'Logistic Reg':>15}")
print("-"*60)
print(f"{'Accuracy':<20} {rf_results['accuracy']:>15.4f} {lr_results['accuracy']:>15.4f}")
print(f"{'Precision':<20} {rf_results['weightedPrecision']:>15.4f} {lr_results['weightedPrecision']:>15.4f}")
print(f"{'Recall':<20} {rf_results['weightedRecall']:>15.4f} {lr_results['weightedRecall']:>15.4f}")
print(f"{'F1 Score':<20} {rf_results['f1']:>15.4f} {lr_results['f1']:>15.4f}")
print("="*60)

# ── 8. Confusion Matrix ──────────────────────────────────
print("\n📊 CONFUSION MATRIX — Random Forest:")
print("(0=NO RISK  1=HIGH  2=CRITICAL)")
rf_preds.groupBy("label", "prediction").count() \
    .orderBy("label", "prediction").show()

# ── 9. Feature Importances ───────────────────────────────
print("\n🌲 TOP FEATURES DRIVING FIRE RISK:")
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.stages[-1].featureImportances.toArray()
}).sort_values("importance", ascending=False)

for _, row in importance_df.iterrows():
    bar = "█" * int(row["importance"] * 100)
    print(f"{row['feature']:<35} {row['importance']:.4f}  {bar}")

# ── 10. Save Predictions to Gold ─────────────────────────
gold = spark.table("sf_fire_gold.property_risk_master").fillna(0)
if "features" in gold.columns:
    gold = gold.drop("features")

final_preds = rf_model.transform(gold)
final_output = final_preds.select(
    "address_clean", "risk_score", "risk_tier",
    "risk_category", "risk_label",
    col("prediction").alias("ml_prediction"),
    col("probability").alias("ml_probability")
).withColumn("ml_risk_category",
    when(col("ml_prediction") == 2, "CRITICAL")
    .when(col("ml_prediction") == 1, "HIGH")
    .otherwise("NO RISK")
)

final_output.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("sf_fire_gold.ml_risk_predictions")

print("\n✅ Predictions saved to sf_fire_gold.ml_risk_predictions")

# ── Save Models to Unity Catalog Volume ──────────────────
RF_PATH = "dbfs:/Volumes/workspace/default/models/sf_fire_rf_model"
LR_PATH = "dbfs:/Volumes/workspace/default/models/sf_fire_lr_model"

# Create volume first
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.default.models
""")

# Save models
rf_model.write().overwrite().save(RF_PATH)
lr_model.write().overwrite().save(LR_PATH)

print(f"✅ Random Forest saved → {RF_PATH}")
print(f"✅ Logistic Regression saved → {LR_PATH}")